In [ ]:
import os
import re
import json
import math
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =========================================================
# CONFIG
# =========================================================
DATA_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecast_data"
TEST_JSONL = os.path.join(DATA_DIR, "test.jsonl")

MODEL_DIR = "/content/drive/MyDrive/FOA_Data/llm_forecaster_model_multi"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MAX_INPUT_LEN = 1024
MAX_NEW_TOKENS = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# LOAD MODEL
# =========================================================
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.to(DEVICE)
model.eval()

# =========================================================
# HELPERS
# =========================================================
def parse_numbers(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return [float(x) for x in nums]

def mae(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def rmse(y_true, y_pred):
    return math.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    eps = 1e-8
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

# =========================================================
# LOAD TEST DATA
# =========================================================
rows = []
with open(TEST_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print("Test examples:", len(rows))

all_true = []
all_pred = []
results = []
fail_count = 0

# =========================================================
# RUN INFERENCE
# =========================================================
for i, row in enumerate(rows):
    prompt = (
        "### Instruction:\n"
        f"{row['prompt']}\n\n"
        "### Response:\n"
    )

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    generated = decoded.split("### Response:\n")[-1].strip()

    pred_nums = parse_numbers(generated)
    true_nums = parse_numbers(row["target"])

    if len(true_nums) != 1:
        continue

    true_val = true_nums[0]

    if len(pred_nums) == 0:
        fail_count += 1
        pred_val = 0.0
    else:
        pred_val = pred_nums[0]

    all_true.append(true_val)
    all_pred.append(pred_val)

    results.append({
        "ticker": row.get("ticker", ""),
        "date": row.get("date", ""),
        "true": true_val,
        "pred": pred_val,
        "generated_text": generated
    })

    if i % 10 == 0:
        print("\n--- Example", i, "---")
        print("Ticker:", row.get("ticker", ""))
        print("Date  :", row.get("date", ""))
        print("Target:", row["target"])
        print("Pred  :", f"{pred_val:.3f}")
        print("Raw   :", generated)

# =========================================================
# METRICS
# =========================================================
print("\nFailures needing fallback:", fail_count)
print("MAE :", mae(all_true, all_pred))
print("RMSE:", rmse(all_true, all_pred))
print("MAPE:", mape(all_true, all_pred))

results_df = pd.DataFrame(results)
results_path = os.path.join(DATA_DIR, "test_predictions.csv")
results_df.to_csv(results_path, index=False)

print("Saved predictions to:", results_path)
results_df.head()